# Day 12 — Treatment Assignment and Causal Identification

## Project

EconoCausal — Dynamic Pricing via Double Machine Learning

## Objectives

* Investigate potential treatment-assignment variables.
* Examine campaign and coupon information.
* Determine whether treatment assignment can be identified from the available data.
* Investigate the limitations of using transaction discounts as treatment variables.
* Document the assumptions required for causal analysis.

## Expected Outcome

A preliminary assessment of whether the available datasets support a defensible treatment definition for causal inference.


In [4]:
import pandas as pd
import numpy as np

# Load datasets
train = pd.read_csv("../data/raw/train.csv")

transactions = pd.read_csv(
    "../data/raw/customer_transaction_data.csv"
)

campaigns = pd.read_csv(
    "../data/raw/campaign_data.csv"
)

customers = pd.read_csv(
    "../data/raw/customer_demographics.csv"
)

coupon_item_mapping = pd.read_csv(
    "../data/raw/coupon_item_mapping.csv"
)

print("All datasets loaded successfully!")

print("Train:", train.shape)
print("Transactions:", transactions.shape)
print("Campaigns:", campaigns.shape)
print("Customers:", customers.shape)
print("Coupon-item mapping:", coupon_item_mapping.shape)

All datasets loaded successfully!
Train: (78369, 5)
Transactions: (1324566, 7)
Campaigns: (28, 4)
Customers: (760, 7)
Coupon-item mapping: (92663, 2)


In [5]:
# Inspect campaign and coupon information

print("===== CAMPAIGN DATASET =====")
print("Columns:")
print(campaigns.columns.tolist())

display(campaigns.head())

print("\n===== TRAIN DATASET =====")
print("Columns:")
print(train.columns.tolist())

display(train.head())

print("\n===== COUPON-ITEM MAPPING DATASET =====")
coupon_item_mapping = pd.read_csv(
    "../data/raw/coupon_item_mapping.csv"
)

print("Columns:")
print(coupon_item_mapping.columns.tolist())

display(coupon_item_mapping.head())

===== CAMPAIGN DATASET =====
Columns:
['campaign_id', 'campaign_type', 'start_date', 'end_date']


,campaign_id,campaign_type,start_date,end_date
0,24,Y,21/10/13,20/12/13
1,25,Y,21/10/13,22/11/13
2,20,Y,07/09/13,16/11/13
3,23,Y,08/10/13,15/11/13
4,21,Y,16/09/13,18/10/13



===== TRAIN DATASET =====
Columns:
['id', 'campaign_id', 'coupon_id', 'customer_id', 'redemption_status']


,id,campaign_id,coupon_id,customer_id,redemption_status
0,1,13,27,1053,0
1,2,13,116,48,0
2,6,9,635,205,0
3,7,13,644,1050,0
4,9,8,1017,1489,0



===== COUPON-ITEM MAPPING DATASET =====
Columns:
['coupon_id', 'item_id']


,coupon_id,item_id
0,105,37
1,107,75
2,494,76
3,522,77
4,518,77


In [6]:
# Search for treatment-related variables

datasets = {
    "Train": train,
    "Campaigns": campaigns,
    "Transactions": transactions,
    "Customers": customers,
    "Coupon-Item Mapping": coupon_item_mapping
}

keywords = [
    "discount",
    "price",
    "offer",
    "treatment",
    "campaign",
    "coupon",
    "redeem",
    "exposure",
    "eligibility",
    "assignment",
    "control",
    "target"
]

for dataset_name, df in datasets.items():
    print(f"\n===== {dataset_name.upper()} =====")

    matching_columns = [
        col for col in df.columns
        if any(keyword in col.lower() for keyword in keywords)
    ]

    print("Potentially relevant columns:", matching_columns)

    if not matching_columns:
        print("No matching columns found.")


===== TRAIN =====
Potentially relevant columns: ['campaign_id', 'coupon_id']

===== CAMPAIGNS =====
Potentially relevant columns: ['campaign_id', 'campaign_type']

===== TRANSACTIONS =====
Potentially relevant columns: ['selling_price', 'other_discount', 'coupon_discount']

===== CUSTOMERS =====
Potentially relevant columns: []
No matching columns found.

===== COUPON-ITEM MAPPING =====
Potentially relevant columns: ['coupon_id']


In [7]:
# Inspect campaign and coupon information

print("===== CAMPAIGN DATASET =====")
print("Columns:", campaigns.columns.tolist())
display(campaigns.head())

print("\n===== TRAIN DATASET =====")
print("Columns:", train.columns.tolist())
display(train.head())

print("\n===== COUPON-ITEM MAPPING DATASET =====")
print("Columns:", coupon_item_mapping.columns.tolist())
display(coupon_item_mapping.head())

===== CAMPAIGN DATASET =====
Columns: ['campaign_id', 'campaign_type', 'start_date', 'end_date']


,campaign_id,campaign_type,start_date,end_date
0,24,Y,21/10/13,20/12/13
1,25,Y,21/10/13,22/11/13
2,20,Y,07/09/13,16/11/13
3,23,Y,08/10/13,15/11/13
4,21,Y,16/09/13,18/10/13



===== TRAIN DATASET =====
Columns: ['id', 'campaign_id', 'coupon_id', 'customer_id', 'redemption_status']


,id,campaign_id,coupon_id,customer_id,redemption_status
0,1,13,27,1053,0
1,2,13,116,48,0
2,6,9,635,205,0
3,7,13,644,1050,0
4,9,8,1017,1489,0



===== COUPON-ITEM MAPPING DATASET =====
Columns: ['coupon_id', 'item_id']


,coupon_id,item_id
0,105,37
1,107,75
2,494,76
3,522,77
4,518,77


In [8]:
# Search for treatment-related variables

datasets = {
    "Train": train,
    "Campaigns": campaigns,
    "Transactions": transactions,
    "Customers": customers,
    "Coupon-Item Mapping": coupon_item_mapping
}

keywords = [
    "discount", "price", "offer", "treatment",
    "campaign", "coupon", "redeem", "exposure",
    "eligibility", "assignment", "control", "target"
]

for dataset_name, df in datasets.items():
    print(f"\n===== {dataset_name.upper()} =====")

    matching_columns = [
        col for col in df.columns
        if any(word in col.lower() for word in keywords)
    ]

    print("Potentially relevant columns:", matching_columns)


===== TRAIN =====
Potentially relevant columns: ['campaign_id', 'coupon_id']

===== CAMPAIGNS =====
Potentially relevant columns: ['campaign_id', 'campaign_type']

===== TRANSACTIONS =====
Potentially relevant columns: ['selling_price', 'other_discount', 'coupon_discount']

===== CUSTOMERS =====
Potentially relevant columns: []

===== COUPON-ITEM MAPPING =====
Potentially relevant columns: ['coupon_id']


In [9]:
# Display all campaign records
print("Campaign dataset columns:")
print(campaigns.columns.tolist())

print("\nCampaign dataset information:")
campaigns.info()

print("\nFirst 10 campaign records:")
display(campaigns.head(10))

print("\nUnique values in categorical campaign columns:")

for col in campaigns.select_dtypes(
    include=["object", "category"]
).columns:
    print(f"\n{col}:")
    print(campaigns[col].value_counts(dropna=False).head(10))

Campaign dataset columns:
['campaign_id', 'campaign_type', 'start_date', 'end_date']

Campaign dataset information:
<class 'pandas.DataFrame'>
RangeIndex: 28 entries, 0 to 27
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   campaign_id    28 non-null     int64
 1   campaign_type  28 non-null     str  
 2   start_date     28 non-null     str  
 3   end_date       28 non-null     str  
dtypes: int64(1), str(3)
memory usage: 1.5 KB

First 10 campaign records:


,campaign_id,campaign_type,start_date,end_date
0,24,Y,21/10/13,20/12/13
1,25,Y,21/10/13,22/11/13
2,20,Y,07/09/13,16/11/13
3,23,Y,08/10/13,15/11/13
4,21,Y,16/09/13,18/10/13
5,22,X,16/09/13,18/10/13
6,18,X,10/08/13,04/10/13
7,19,Y,26/08/13,27/09/13
8,17,Y,29/07/13,30/08/13
9,16,Y,15/07/13,16/08/13



Unique values in categorical campaign columns:

campaign_type:
campaign_type
Y    22
X     6
Name: count, dtype: int64

start_date:
start_date
21/10/13    2
16/09/13    2
22/04/13    2
07/09/13    1
08/10/13    1
10/08/13    1
26/08/13    1
29/07/13    1
15/07/13    1
19/05/13    1
Name: count, dtype: int64

end_date:
end_date
18/10/13    2
18/01/13    2
20/12/13    1
22/11/13    1
16/11/13    1
15/11/13    1
04/10/13    1
27/09/13    1
30/08/13    1
16/08/13    1
Name: count, dtype: int64


C:\Users\KESHAV DEV GAUR\AppData\Local\Temp\ipykernel_25312\2528775078.py:13: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in campaigns.select_dtypes(


In [10]:
# Merge training records with campaign information
campaign_analysis = train.merge(
    campaigns,
    on="campaign_id",
    how="left",
    validate="many_to_one"
)

# Summarize redemption by campaign type
campaign_type_summary = (
    campaign_analysis
    .groupby("campaign_type", dropna=False)
    .agg(
        total_records=("redemption_status", "size"),
        total_redemptions=("redemption_status", "sum"),
        redemption_rate=("redemption_status", "mean"),
        unique_campaigns=("campaign_id", "nunique")
    )
    .reset_index()
)

campaign_type_summary["redemption_rate_pct"] = (
    campaign_type_summary["redemption_rate"] * 100
).round(2)

display(campaign_type_summary)

,campaign_type,total_records,total_redemptions,redemption_rate,unique_campaigns,redemption_rate_pct
0,X,57684,636,0.011026,4,1.10
1,Y,20685,93,0.004496,14,0.45


In [11]:
# Inspect treatment-related columns in the training dataset

print("Training dataset columns:")
print(train.columns.tolist())

print("\nPotential assignment and exposure columns:")

keywords = [
    "treatment",
    "assignment",
    "exposure",
    "eligibility",
    "control",
    "discount",
    "offer",
    "campaign",
    "coupon"
]

for col in train.columns:
    if any(keyword in col.lower() for keyword in keywords):
        print(f"\nColumn: {col}")
        print("Unique values:", train[col].nunique(dropna=False))
        print("Missing values:", train[col].isna().sum())
        print("Sample values:")
        print(train[col].head(5).tolist())

Training dataset columns:
['id', 'campaign_id', 'coupon_id', 'customer_id', 'redemption_status']

Potential assignment and exposure columns:

Column: campaign_id
Unique values: 18
Missing values: 0
Sample values:
[13, 13, 9, 13, 8]

Column: coupon_id
Unique values: 866
Missing values: 0
Sample values:
[27, 116, 635, 644, 1017]


In [12]:
import pandas as pd

# Preliminary treatment assignment assessment
treatment_assessment = pd.DataFrame([
    {
        "question": "Is campaign identity available?",
        "evidence": "campaign_id exists in campaign_data and train",
        "assessment": "Available",
        "limitation": "An identifier does not prove customer exposure."
    },
    {
        "question": "Are campaign dates available?",
        "evidence": "start_date and end_date exist in campaign_data",
        "assessment": "Available",
        "limitation": "Dates do not establish which customers received an offer."
    },
    {
        "question": "Is campaign type available?",
        "evidence": "campaign_type contains X and Y",
        "assessment": "Available",
        "limitation": "The meaning of X and Y must be verified before interpretation."
    },
    {
        "question": "Is coupon redemption recorded?",
        "evidence": "redemption_status exists in train",
        "assessment": "Available",
        "limitation": "Redemption is an outcome, not treatment assignment."
    },
    {
        "question": "Is assigned discount treatment verified?",
        "evidence": "No verified assignment field established so far",
        "assessment": "Unresolved",
        "limitation": "Additional documentation or assignment data may be required."
    },
    {
        "question": "Can a causal effect be claimed?",
        "evidence": "Treatment assignment and identification assumptions remain unverified",
        "assessment": "Not yet",
        "limitation": "Do not interpret redemption differences as causal effects."
    }
])

display(treatment_assessment)

,question,evidence,assessment,limitation
0,Is campaign identity available?,campaign_id exists in campaign_data and train,Available,An identifier does not prove customer exposure.
1,Are campaign dates available?,start_date and end_date exist in campaign_data,Available,Dates do not establish which customers receive...
2,Is campaign type available?,campaign_type contains X and Y,Available,The meaning of X and Y must be verified before...
3,Is coupon redemption recorded?,redemption_status exists in train,Available,"Redemption is an outcome, not treatment assign..."
4,Is assigned discount treatment verified?,No verified assignment field established so far,Unresolved,Additional documentation or assignment data ma...
5,Can a causal effect be claimed?,Treatment assignment and identification assump...,Not yet,Do not interpret redemption differences as cau...


In [13]:
# Save the treatment assignment assessment
treatment_assessment.to_csv(
    "../reports/day12_treatment_assignment_assessment.csv",
    index=False
)

print("Day 12 treatment assessment saved successfully!")

Day 12 treatment assessment saved successfully!
